# 🚀 Фабрика нейросетей & Асинхронный Walk-Forward рой PRL/USDT (Google Colab / GPU T4)
Весь тяжёлый код архитектур и обучения вынесен в модульные скрипты  и .
Управляйте параметрами обучения напрямую через аргументы в ячейках ниже.

In [ ]:
!nvidia-smi
import os, sys, importlib
if os.path.exists("/content"):
    !mkdir -p _tools
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_models.py -o _tools/colab_models.py
    !curl -sL https://raw.githubusercontent.com/BulletproofBoris/crypto_bot/main/_tools/colab_neural_trainer.py -o _tools/colab_neural_trainer.py
sys.path.insert(0, os.path.abspath("."))
import _tools.colab_models, _tools.colab_neural_trainer
importlib.reload(_tools.colab_models)
importlib.reload(_tools.colab_neural_trainer)
from _tools.colab_neural_trainer import run_colab_pipeline
print("✅ Свежие модули _tools.colab_models и _tools.colab_neural_trainer загружены!")

In [ ]:
# 2. Конфигурация и запуск Walk-Forward обучения нейросетей (H=60с, K=3.0, L=180с)
# После КАЖДОГО рана автоматически сохраняются и выгружаются в облако:
#   - лучшие веса model_win_00.keras + метаданные meta_win_00.json
#   - состояние ранов progress_win_00.json + лог training_log_h60s.txt
# При перезапуске или переносе на другой сервер обучение автоматически продолжится со следующего рана!
import os, shutil
FILE_ID = "1PeblLTnFKL3tCy-1bhQt23B-kwUt34-k"              # Свежий датасет от 28.10 (H=5: 36, H=12: 12, H=25: 6, H=60: 1)
HORIZON = 5                                                # Горизонт H (сек): 5, 12 или 25 (только новое переднее окно win_00!)
WINDOWS = "0"                                              # Строгое 100%-OOS новое окно win_00
ARCHS = "conv1d+gru,causal_cnn,attention,mlp_bottleneck"   # Все 4 архитектуры через запятую
RUNS_PER_ARCH = 10                                         # Число независимых ранов на архитектуру (синхронизировано с ночным циклом)
MAX_EPOCHS = 10                                            # Макс. эпох на один ран
BATCH_SIZE = 2048                                          # Размер батча на GPU T4
LR = 2e-3                                                  # Скорость обучения AdamW
SYNC_BUCKET = f"prl_h{HORIZON}s_win00_2810"               # Постоянный облачный бакет авто-сохранения после каждого рана
RESUME_URL = f"https://filebin.net/{SYNC_BUCKET}/prl_ensemble_pack_h{HORIZON}s.zip"

# В Colab очищаем старый локальный кэш пакета, чтобы гарантированно подтянуть свежий ночной чекпоинт (r07/10)
if os.path.exists("/content"):
    shutil.rmtree(f"prl_ensemble_pack_h{HORIZON}s", ignore_errors=True)
    if os.path.exists(f"prl_ensemble_pack_h{HORIZON}s.zip"):
        os.remove(f"prl_ensemble_pack_h{HORIZON}s.zip")

df_leaderboard = run_colab_pipeline(
    file_id=FILE_ID,
    horizon=HORIZON, windows=WINDOWS, archs=ARCHS,
    runs_per_arch=RUNS_PER_ARCH, max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE, lr=LR,
    sync_bucket=SYNC_BUCKET, resume_url=RESUME_URL,
)


In [ ]:
# 3. Выгрузка готового архива моделей (авто-упаковка при прерванной 2-й ячейке)
import os, shutil
HORIZON = globals().get("HORIZON", 60)
!pkill -9 -f "colab_neural_trainer" 2>/dev/null || true
PACK_DIR = f"prl_ensemble_pack_h{HORIZON}s"
ZIP_NAME = f"{PACK_DIR}.zip"
if os.path.exists(PACK_DIR):
    shutil.make_archive(PACK_DIR, "zip", root_dir=".", base_dir=PACK_DIR)
if os.path.exists(ZIP_NAME):
    print(f"📦 Архив найден: {ZIP_NAME} ({os.path.getsize(ZIP_NAME) / (1024*1024):.2f} MB)")
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://x0.at/ | head -c 200 && echo " (x0.at)"
    !curl -s --max-time 60 -F "file=@{ZIP_NAME}" https://store1.gofile.io/uploadFile | head -c 400 && echo " (gofile)"
    !curl -s --max-time 60 -H "Accept: application/json" -T "{ZIP_NAME}" https://filebin.net/prl_h{HORIZON}s_pack/{ZIP_NAME} | head -c 300 && echo " (filebin)"
